# 02 - Fashion CNN V0 - Inspection dataset

Objectif : preparer Fashion Product Images Small pour un futur CNN V0 sans entrainer de modele dans ce notebook.

La cible future est `product_type_v0`, derivee de `styles.csv.articleType`. Le modele predira un type produit visible, puis `canonical_category` sera derivee pour le moteur outfit.

Ce notebook s'arrete explicitement avant entrainement tant que `config/fashion_v1_classes.json` reste en statut `draft_requires_dataset_inspection`.


## 1. Monter Google Drive


In [ ]:
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path('/content/drive')
if (DRIVE_MOUNT / 'MyDrive').exists():
    print('Google Drive deja monte.')
else:
    drive.mount(str(DRIVE_MOUNT))

DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive'
print(f'Drive root: {DRIVE_ROOT}')


## 2. Cloner ou mettre a jour le repo GitHub


In [ ]:
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/MilFhey/fit-outfit-advisor.git'
BRANCH = 'main'
REPO_DIR = Path('/content/fit-outfit-advisor-repo')
PROJECT_DIR = REPO_DIR / 'fit-outfit-advisor'

if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('Repo existant : mise a jour.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=str(REPO_DIR), check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=str(REPO_DIR), check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Dossier projet absent dans le repo clone : {PROJECT_DIR}')

sys.path.insert(0, str(PROJECT_DIR))
print(f'Repo pret : {REPO_DIR}')
print(f'Projet pret : {PROJECT_DIR}')


## 3. Installer les dependances


In [ ]:
requirements_path = PROJECT_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError(f'Requirements absent : {requirements_path}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kaggle'], check=True)


## 4. Creer les dossiers temporaires


In [ ]:
RUNTIME_ROOT = Path('/content/fit-outfit-runtime')
KAGGLE_DOWNLOAD_DIR = RUNTIME_ROOT / 'kaggle_downloads'
CONTENT_DATA_DIR = RUNTIME_ROOT / 'data'
CONTENT_ARTIFACT_DIR = RUNTIME_ROOT / 'artifacts'

for directory in [RUNTIME_ROOT, KAGGLE_DOWNLOAD_DIR, CONTENT_DATA_DIR, CONTENT_ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
    print(directory)


## 5. Charger le Secret Colab Kaggle

Le Secret Colab attendu s'appelle exactement `KAGGLE_API`.


In [ ]:
from google.colab import userdata

kaggle_token = userdata.get('KAGGLE_API')
if not kaggle_token:
    raise ValueError('Secret Colab introuvable : cree ou autorise le Secret KAGGLE_API.')
if not str(kaggle_token).startswith('KGAT_'):
    raise ValueError('Le Secret KAGGLE_API semble invalide : il doit commencer par KGAT_.')

os.environ['KAGGLE_API_TOKEN'] = kaggle_token
os.environ['KAGGLE_API'] = kaggle_token
print('Token Kaggle charge depuis KAGGLE_API.')


## 6. Telecharger Fashion Product Images Small


In [ ]:
KAGGLE_DATASET = 'paramaggarwal/fashion-product-images-small'

existing_files = [path for path in KAGGLE_DOWNLOAD_DIR.rglob('*') if path.is_file()]
if existing_files:
    print(f'Dataset deja present dans {KAGGLE_DOWNLOAD_DIR} ({len(existing_files)} fichiers).')
else:
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(KAGGLE_DOWNLOAD_DIR), '--unzip'],
        check=True,
    )

downloaded_files = sorted(path for path in KAGGLE_DOWNLOAD_DIR.rglob('*') if path.is_file())
print(f'Fichiers detectes : {len(downloaded_files)}')
for path in downloaded_files[:20]:
    print(path.relative_to(KAGGLE_DOWNLOAD_DIR))


## 7. Detecter `styles.csv` et le dossier `images/`


In [ ]:
styles_candidates = sorted(KAGGLE_DOWNLOAD_DIR.rglob('styles.csv'))
if not styles_candidates:
    styles_candidates = sorted(KAGGLE_DOWNLOAD_DIR.rglob('*.csv'))
if not styles_candidates:
    raise FileNotFoundError('Aucun CSV de metadata trouve dans le dataset Kaggle.')

STYLES_CSV = styles_candidates[0]

image_dir_candidates = [path for path in KAGGLE_DOWNLOAD_DIR.rglob('images') if path.is_dir()]
if not image_dir_candidates:
    image_dir_candidates = sorted({path.parent for path in KAGGLE_DOWNLOAD_DIR.rglob('*.jpg')})
if not image_dir_candidates:
    raise FileNotFoundError('Aucun dossier images ou fichier .jpg trouve.')

IMAGE_DIR = image_dir_candidates[0]
print(f'STYLES_CSV = {STYLES_CSV}')
print(f'IMAGE_DIR = {IMAGE_DIR}')
print(f'Nombre de .jpg detectes = {len(list(IMAGE_DIR.glob("*.jpg")))}')


## 8. Inspecter les metadata


In [ ]:
import pandas as pd

df = pd.read_csv(STYLES_CSV, on_bad_lines='skip')
print('shape =', df.shape)
print('colonnes =')
print(list(df.columns))

display(df.head())

missing = (
    df.isna().sum()
    .rename('missing_count')
    .to_frame()
)
missing['missing_pct'] = (missing['missing_count'] / len(df) * 100).round(2)
display(missing.sort_values('missing_count', ascending=False))

for column in ['masterCategory', 'subCategory', 'articleType', 'baseColour', 'usage', 'gender']:
    if column in df.columns:
        print(f'\nDistribution {column}')
        display(df[column].value_counts(dropna=False).head(40).to_frame('count'))


## 9. Charger la configuration de classes et proposer une categorie canonique

Les propositions ci-dessous ne modifient pas `config/fashion_v1_classes.json`. Elles servent uniquement a guider la revue humaine apres inspection.


In [ ]:
import importlib
import json
import sys

PROJECT_DIR_STR = str(PROJECT_DIR)
sys.path = [PROJECT_DIR_STR] + [path for path in sys.path if path != PROJECT_DIR_STR]
for module_name in list(sys.modules):
    if module_name == 'src' or module_name.startswith('src.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

from src.mappings.fashion_v1_mapping import build_article_type_to_product_type_mapping, load_fashion_v1_class_config, map_product_type_to_canonical_category

CLASS_CONFIG_PATH = PROJECT_DIR / 'config' / 'fashion_v1_classes.json'
class_config = load_fashion_v1_class_config(CLASS_CONFIG_PATH)
configured_mapping = build_article_type_to_product_type_mapping(class_config)

DRAFT_ARTICLE_TYPE_PROPOSALS = {
    'Tshirts': 'tshirt', 'Shirts': 'shirt', 'Tops': 'top', 'Kurtas': 'top', 'Kurtis': 'top', 'Tunics': 'top',
    'Jeans': 'jeans', 'Trousers': 'trousers', 'Track Pants': 'trousers', 'Shorts': 'shorts',
    'Dresses': 'dress',
    'Casual Shoes': 'casual_shoes', 'Sports Shoes': 'sports_shoes', 'Formal Shoes': 'dress_shoes',
    'Sandals': 'sandals', 'Flip Flops': 'flip_flops', 'Heels': 'heels', 'Flats': 'flats',
    'Jackets': 'outerwear', 'Sweaters': 'outerwear', 'Sweatshirts': 'outerwear',
    'Handbags': 'bag', 'Backpacks': 'bag', 'Clutches': 'bag',
    'Watches': 'watch', 'Sunglasses': 'sunglasses', 'Caps': 'cap',
    'Wallets': 'wallet', 'Belts': 'belt',
    'Earrings': 'jewellery', 'Pendant': 'jewellery', 'Necklace and Chains': 'jewellery',
}

def proposed_product_type(article_type):
    article_type = str(article_type).strip()
    if article_type in configured_mapping:
        return configured_mapping[article_type]
    return DRAFT_ARTICLE_TYPE_PROPOSALS.get(article_type)

def proposed_canonical_category(article_type):
    product_type = proposed_product_type(article_type)
    if product_type is None:
        return None
    return map_product_type_to_canonical_category(product_type, class_config)

print(json.dumps(class_config, indent=2, ensure_ascii=False))
print(f'Nombre articleType configures vers product_type_v0 : {len(configured_mapping)}')


## 10. Verifier images presentes et lisibles, puis produire le tableau final


In [ ]:
from PIL import Image

required_columns = {'id', 'articleType'}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f'Colonnes obligatoires absentes : {sorted(missing_columns)}')

audit_df = df[['id', 'articleType']].copy()
audit_df['articleType'] = audit_df['articleType'].astype(str).str.strip()
audit_df['image_path'] = audit_df['id'].astype(str).str.replace(r'\\.0$', '', regex=True).map(lambda image_id: IMAGE_DIR / f'{image_id}.jpg')
audit_df['image_present'] = audit_df['image_path'].map(Path.exists)

def is_readable_image(path):
    if not path.exists():
        return False
    try:
        with Image.open(path) as image:
            image.verify()
        return True
    except Exception:
        return False

audit_df['image_readable'] = audit_df['image_path'].map(is_readable_image)

summary = audit_df.groupby('articleType').agg(
    metadata_row_count=('articleType', 'size'),
    present_image_count=('image_present', 'sum'),
    readable_image_count=('image_readable', 'sum'),
).reset_index()
summary['proposed_product_type_v0'] = summary['articleType'].map(proposed_product_type)
summary['proposed_canonical_category'] = summary['articleType'].map(proposed_canonical_category)

minimum_count = class_config.get('minimum_readable_images_per_class')
config_status = class_config.get('status')

def decide(row):
    article_type = row['articleType']
    if article_type not in configured_mapping:
        return 'exclure', 'articleType absent du mapping product_type_v0 valide'
    if config_status == 'draft_requires_dataset_inspection':
        return 'exclure', 'configuration en brouillon apres inspection requise'
    if minimum_count is None:
        return 'exclure', 'seuil minimal non renseigne'
    if int(row['readable_image_count']) < int(minimum_count):
        return 'exclure', 'images lisibles sous le seuil minimal'
    return 'garder', ''

decisions = summary.apply(decide, axis=1, result_type='expand')
summary['decision'] = decisions[0]
summary['exclusion_reason'] = decisions[1]
summary = summary.sort_values(['decision', 'readable_image_count'], ascending=[True, False])

audit_output = CONTENT_ARTIFACT_DIR / 'fashion_v1_article_type_audit.csv'
summary.to_csv(audit_output, index=False)
print(f'Tableau final ecrit : {audit_output}')
display(summary)

print('Comptage images manquantes :', int((~audit_df['image_present']).sum()))
print('Comptage images corrompues/non lisibles :', int((audit_df['image_present'] & ~audit_df['image_readable']).sum()))


## 11. Apercu visuel d'images reelles


In [ ]:
import matplotlib.pyplot as plt

sample_df = audit_df[audit_df['image_readable']].copy()
if sample_df.empty:
    raise ValueError('Aucune image lisible pour afficher un apercu visuel.')

sample_df['proposed_product_type_v0'] = sample_df['articleType'].map(proposed_product_type)
sample_df['proposed_canonical_category'] = sample_df['articleType'].map(proposed_canonical_category)
sample_df = sample_df.sample(n=min(12, len(sample_df)), random_state=42)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for ax, (_, row) in zip(axes.flatten(), sample_df.iterrows()):
    image = Image.open(row['image_path']).convert('RGB')
    ax.imshow(image)
    ax.set_title(f"{row['articleType']} -> {row['proposed_product_type_v0']} -> {row['proposed_canonical_category']}", fontsize=9)
    ax.axis('off')
for ax in axes.flatten()[len(sample_df):]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## 12. Arret explicite avant entrainement

Ne pas continuer vers l'entrainement tant que le mapping `config/fashion_v1_classes.json` n'a pas ete valide manuellement apres lecture du tableau final.


In [ ]:
if class_config.get('status') == 'draft_requires_dataset_inspection':
    raise RuntimeError(
        'ARRET AVANT ENTRAINEMENT : inspecte le tableau final, puis renseigne '
        'config/fashion_v1_classes.json avec les articleType retenus et le seuil minimal.'
    )

print('Configuration non draft detectee. L entrainement devra etre lance dans une etape separee.')
